# Week 3 · Human-in-the-loop — 업무를 멈추고, 확인하고, 안전하게 재개하기

이 노트북은 Week 1의 **State·Node·Edge**와 Week 2의 **Message·Reducer·Tool 흐름**을 실제 업무 승인 흐름에 연결합니다. 대상 독자는 파이썬의 변수, 함수, 조건문, 리스트와 딕셔너리를 배운 경영정보학과 학회원입니다.

이번 주가 끝나면 다음을 할 수 있어야 합니다.

1. Reducer와 Checkpointer가 서로 다른 질문에 답한다는 것을 설명한다.
2. 같은 thread와 다른 thread에서 State가 어떻게 이어지거나 분리되는지 확인한다.
3. 메모리를 **공유 범위**와 **저장 지속성**이라는 두 축으로 분류한다.
4. 동적 interrupt로 승인·거부를 받고, 위험한 부수효과를 승인 뒤에 둔다.
5. (보충) updates와 values 스트리밍을 읽어 실행 과정을 관찰한다.

**핵심 실습에는 LLM과 OpenAI API 키가 필요하지 않습니다.** 네트워크나 모델 응답의 변동 때문에 LangGraph 원리를 놓치지 않도록 모든 결과를 결정적으로 만들었습니다.

## 이번 주의 업무 사례: 대학 학생지원 문의

한 학생이 장학금 지급일을 문의했다고 가정합니다. 단순 조회는 자동화할 수 있지만, 학생에게 공식 안내를 발송하거나 학사 시스템의 값을 바꾸는 행동은 담당자의 확인이 필요합니다.

~~~text
문의 접수 → 안내문 준비 → 담당자 검토 ─┬→ 승인 → 안내 발송 기록
                                      ├→ 거부 → 처리 중단
                                      └→ 잘못된 입력 → 안전하게 중단
~~~

경영정보시스템 관점에서 이 흐름은 익숙한 결재 프로세스입니다.

- State는 한 업무 건의 전자 문서입니다.
- Node는 접수, 검토, 발송 같은 업무 단계입니다.
- Edge는 다음 업무 단계로 보내는 규칙입니다.
- thread_id는 업무 건을 구분하는 접수번호입니다.
- Checkpoint는 특정 시점의 업무 문서 사본입니다.
- interrupt는 담당자의 결재를 기다리는 대기 상태입니다.

## 학습 방법: 예측 → 실행 → 설명 → 빈칸 → 전이

각 핵심 개념은 다음 순서로 학습합니다.

1. **예측:** 코드를 실행하기 전에 State와 다음 Node를 예상합니다.
2. **실행:** 셀을 위에서 아래로 실행합니다.
3. **설명:** 출력이 왜 그렇게 나왔는지 표와 타임라인으로 확인합니다.
4. **빈칸:** 용어를 스스로 회상합니다.
5. **전이:** 학생지원 사례를 다른 업무 프로세스로 바꿔 설계합니다.

코드가 실행된 것만으로 학습이 끝난 것은 아닙니다. 각 절의 “내 말로 설명하기” 질문에 두 문장으로 답할 수 있어야 합니다.

## 3-0. 준비

Week 0에서 만든 가상환경과 Jupyter 커널을 선택한 뒤 아래 셀을 실행합니다. 이 노트북은 환경변수나 API 키를 읽지 않고, 추가 패키지를 다시 설치하지도 않습니다.

아래 셀은 LangGraph가 올바른 커널에 이미 설치되었는지만 확인합니다. 오류가 나면 Week 0의 가상환경 준비를 먼저 완료하세요.

In [ ]:
# Week 0에서 설치한 LangGraph 버전을 확인합니다.
import importlib.metadata as package_metadata

try:
    langgraph_version = package_metadata.version("langgraph")
except package_metadata.PackageNotFoundError as error:
    raise RuntimeError(
        "LangGraph가 없습니다. Week 0에서 가상환경과 패키지 설치를 완료하세요."
    ) from error

print("✅ LangGraph 버전:", langgraph_version)

In [ ]:
# 이번 주 핵심 실습에서 공통으로 사용할 도구입니다.
import operator
import uuid
from typing import Annotated, Literal, NotRequired, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

print("✅ Week 3 핵심 도구 준비 완료")

## 3-1. 먼저 구분하기: State, Reducer, Checkpointer, thread

네 용어는 함께 등장하지만 역할이 다릅니다.

| 개념 | 답하는 질문 | 학생지원 업무 비유 |
|---|---|---|
| State | 지금 이 업무 건이 가진 값은 무엇인가? | 현재 작성 중인 문의 처리 문서 |
| Reducer | 같은 State 칸에 새 값이 오면 어떻게 합칠까? | 처리 메모를 이어 붙일지, 최신 값으로 덮어쓸지 |
| Checkpointer | 각 단계의 State를 어떻게 저장할까? | 단계가 끝날 때 문서 사본을 남기는 기록 시스템 |
| Checkpoint | 한 시점에 저장된 State 사본은 무엇인가? | “접수 완료 시점”의 문서 버전 |
| thread_id | 어느 실행의 checkpoint를 불러올까? | 문의 접수번호 또는 업무 case ID |

Reducer와 Checkpointer는 경쟁 관계가 아닙니다. **Reducer가 업데이트를 합쳐 현재 State를 만들고, Checkpointer가 그 결과의 스냅샷을 thread별로 저장합니다.**

### 하나의 Node가 끝날 때 벌어지는 일

~~~text
이전 State
   + Node가 반환한 업데이트
   ↓
Reducer가 칸별 결합 규칙 적용
   ↓
새로운 현재 State
   ↓
Checkpointer가 thread_id 아래 checkpoint로 저장
~~~

예를 들어 notes 칸에 <code>operator.add</code> Reducer가 있으면 새 메모가 기존 목록 뒤에 붙습니다. 반면 contact_count처럼 Reducer가 없는 칸은 Node가 반환한 최신 값으로 덮어씁니다.

Checkpointer가 있다고 모든 thread가 서로 섞이는 것은 아닙니다. 같은 thread_id는 같은 업무 기록을 이어 보고, 다른 thread_id는 별도의 업무 기록을 만듭니다.

### Checkpoint 타임라인

~~~text
thread_id = inquiry-101
  checkpoint 0: 문의 입력
  checkpoint 1: 1차 접촉 기록
  checkpoint 2: 2차 접촉 기록

thread_id = inquiry-202
  checkpoint 0: 다른 문의 입력
  checkpoint 1: 이 문의의 1차 접촉 기록
~~~

체크포인트는 “시간의 흐름”, thread_id는 “업무 건의 경계”를 나타냅니다. 따라서 같은 학생이 여러 문의를 만들더라도 문의별 thread_id를 쓰는 것이 자연스럽습니다. 학생 ID와 thread_id를 무조건 같은 값으로 사용하면 서로 다른 문의 기록이 한 State에 섞일 수 있습니다.

## 3-2. 실험: 같은 thread와 다른 thread

아래 그래프는 LLM 없이 문의 접촉 횟수와 처리 메모를 기록합니다.

**실행 전 예측**

1. 같은 thread_id로 두 번 실행하면 contact_count는 얼마일까요?
2. 두 번째 실행의 notes에는 메모가 몇 개 있을까요?
3. 새로운 thread_id의 첫 실행은 앞 thread의 count를 이어받을까요?

종이에 먼저 적은 뒤 코드를 실행하세요. 정답은 실험 뒤에 확인합니다.

In [ ]:
class ContactHistoryState(TypedDict):
    inquiry: str
    contact_count: NotRequired[int]
    notes: Annotated[NotRequired[list[str]], operator.add]


def record_contact(state: ContactHistoryState):
    next_count = state.get("contact_count", 0) + 1
    return {
        "contact_count": next_count,
        "notes": [f"{next_count}회차 기록: {state['inquiry']}"],
    }

In [ ]:
inquiry_builder = StateGraph(ContactHistoryState)
inquiry_builder.add_node("record_contact", record_contact)
inquiry_builder.add_edge(START, "record_contact")
inquiry_builder.add_edge("record_contact", END)

inquiry_graph = inquiry_builder.compile(checkpointer=InMemorySaver())

run_tag = uuid.uuid4().hex[:8]
same_thread_config = {
    "configurable": {"thread_id": f"inquiry-same-{run_tag}"}
}
new_thread_config = {
    "configurable": {"thread_id": f"inquiry-new-{run_tag}"}
}

In [ ]:
first_result = inquiry_graph.invoke(
    {"inquiry": "장학금 지급일이 궁금합니다."},
    same_thread_config,
)

second_result = inquiry_graph.invoke(
    {"inquiry": "지급 예정일을 다시 확인하고 싶습니다."},
    same_thread_config,
)

print("첫 실행 count:", first_result["contact_count"])
print("두 번째 실행 count:", second_result["contact_count"])
print("두 번째 실행 notes:", second_result["notes"])

assert second_result["contact_count"] == 2
assert len(second_result["notes"]) == 2

### 실행 결과 설명

| 실행 | 사용한 thread_id | 실행 전 State | Node 실행 뒤 |
|---|---|---|---|
| 첫 실행 | same | 새 State | count 1, 메모 1개 |
| 두 번째 실행 | same | 첫 실행의 checkpoint에서 시작 | count 2, 메모 2개 |

첫 입력에는 inquiry만 주었고 contact_count와 notes는 Node가 처음 만듭니다. 두 번째 입력에도 두 칸을 다시 주지 않았지만, 같은 thread_id의 최신 checkpoint가 먼저 복원되므로 이전 값이 남아 있습니다. record_contact가 반환한 notes 한 개는 <code>operator.add</code> Reducer를 통해 기존 목록 뒤에 붙습니다.

즉, **같은 thread라서 이전 State를 불러왔고, Reducer가 있어서 notes가 누적되었습니다.** 두 원인을 섞어 설명하지 않는 것이 중요합니다.

In [ ]:
fresh_result = inquiry_graph.invoke(
    {"inquiry": "학생증 재발급 절차가 궁금합니다."},
    new_thread_config,
)

print("새 thread의 count:", fresh_result["contact_count"])
print("새 thread의 notes:", fresh_result["notes"])

assert fresh_result["contact_count"] == 1
assert len(fresh_result["notes"]) == 1

In [ ]:
# get_state는 해당 thread의 최신 checkpoint를 보여 줍니다.
latest_snapshot = inquiry_graph.get_state(same_thread_config)

print("최신 State:", latest_snapshot.values)
print("다음 Node:", latest_snapshot.next)

assert latest_snapshot.values["contact_count"] == 2
assert latest_snapshot.next == ()

### 예측 확인과 내 말로 설명하기

<details>
<summary>예측 문제 정답</summary>

1. 같은 thread의 두 번째 실행은 count 2입니다.
2. notes에는 두 개의 메모가 있습니다.
3. 새 thread는 앞 thread를 이어받지 않으므로 첫 실행 뒤 count 1입니다.

</details>

**내 말로 설명하기:** “같은 thread”와 “Reducer가 있는 칸”이 각각 어떤 역할을 했는지 두 문장으로 설명해 보세요.

- 같은 thread_id: 이전 checkpoint를 찾아 같은 업무 State를 이어 줍니다.
- notes Reducer: 기존 notes와 새 notes를 어떤 방식으로 합칠지 정합니다.

### 빈칸 채우기

1. Node의 새 값과 기존 State 값을 합치는 규칙은 ________ 입니다.
2. 단계별 State 사본을 저장하는 구성요소는 ________ 입니다.
3. 같은 업무 기록을 재개하려면 같은 ________ 를 사용해야 합니다.
4. Reducer가 없는 일반 칸은 보통 새 값으로 ________ 됩니다.

<details>
<summary>정답 보기</summary>

1. Reducer
2. Checkpointer
3. thread_id
4. 덮어쓰기

</details>

## 3-3. Memory를 두 축으로 보기

“메모리가 오래 남는다”와 “여러 대화에서 공유된다”는 서로 다른 문제입니다. 이를 한 단어인 장기 메모리로만 부르면 설계가 흐려집니다.

### 축 A · 공유 범위

- **thread 범위:** 한 문의나 한 대화 안에서만 이어지는 State입니다.
- **cross-thread 범위:** 여러 문의에서 함께 사용하는 학생 프로필, 선호도, 정책 지식 등입니다.

### 축 B · 저장 지속성

- **RAM:** 프로세스나 커널을 종료하면 사라집니다.
- **DB/영구 저장소:** 프로세스를 다시 시작해도 남을 수 있습니다.

공유 범위는 “누가 보나?”, 지속성은 “언제까지 남나?”에 답합니다.

### 두 축의 2×2 표

| 공유 범위 \ 지속성 | RAM | DB·영구 저장소 |
|---|---|---|
| 한 thread | InMemorySaver로 하는 이번 실습 | DB Checkpointer에 저장한 실제 문의 처리 기록 |
| cross-thread | 테스트용 InMemoryStore | 여러 문의가 공유하는 학생 프로필 Store |

중요한 결론은 다음과 같습니다.

- SQLite나 다른 DB에 Checkpoint를 저장해도 자동으로 cross-thread 메모리가 되지 않습니다. 여전히 특정 thread의 State입니다.
- InMemoryStore는 여러 thread가 공유할 수 있지만 RAM 기반이면 커널 종료 후 사라집니다.
- 이번 주의 InMemorySaver는 개념 실험용입니다. 실제 승인 대기 업무는 프로세스가 종료되어도 복구할 수 있는 DB Checkpointer가 필요합니다.
- 이 절에서는 축을 구분하는 데 집중하며 DB와 Store 코드는 작성하지 않습니다.

### 개념 분류 연습

다음 사례의 **공유 범위**와 **지속성**을 각각 골라 보세요.

1. 노트북을 켜 둔 동안 한 문의의 승인 상태를 저장한다.
2. 서버 재시작 뒤에도 특정 문의의 승인 기록을 복구한다.
3. 여러 문의에서 같은 학생의 선호 언어를 사용하지만, 테스트가 끝나면 지워도 된다.
4. 여러 학기와 여러 문의에서 학생의 동의 정보를 사용한다.

<details>
<summary>정답 보기</summary>

1. thread 범위 + RAM
2. thread 범위 + DB
3. cross-thread 범위 + RAM
4. cross-thread 범위 + DB

</details>

## 3-4. 동적 interrupt: 담당자의 승인·거부 받기

interrupt는 그래프 실행을 사람의 판단이 필요한 지점에서 멈춥니다. 이번 사례에서 “안내문 준비”는 되돌려도 위험하지 않은 계산이지만, “공식 안내 발송 기록 추가”는 외부 시스템을 바꾸는 **부수효과**입니다.

**부수효과(side effect)**란 함수가 State 계산을 넘어 외부 세계를 바꾸는 일입니다. 예를 들면 메일 발송, 결제, DB 행 추가, 파일 삭제가 있습니다.

~~~text
START → prepare → review(interrupt)
                         │
             ┌───────────┼───────────┐
             ↓           ↓           ↓
          execute      reject      invalid
         부수효과       안전 종료     안전 종료
             └───────────┴───────────┘
                         ↓
                        END
~~~

핵심 규칙은 “승인받은 뒤에 별도의 Node에서 부수효과를 실행한다”입니다.

### pause와 resume의 타임라인

| 시점 | 저장된 State | 호출자에게 보이는 것 |
|---|---|---|
| 최초 입력 | request와 빈 결과 칸 | 그래프 실행 중 |
| prepare 완료 | draft가 추가된 State | 다음 Node로 이동 |
| review에서 interrupt | review 직전까지의 State가 checkpoint에 저장 | interrupt payload가 <code>__interrupt__</code>로 반환 |
| 같은 thread_id로 resume | review Node가 처음부터 다시 시작 | interrupt가 resume 값을 반환 |
| 분기 완료 | decision과 result가 추가된 State | 최종 State |

<code>__interrupt__</code>는 우리가 정의한 State 칸이 아닙니다. 런타임이 “지금 사람의 입력을 기다린다”고 알려 주는 **특별한 출력 키**입니다. 실제 저장 State는 <code>graph.get_state(config).values</code>로 구분해서 확인할 수 있습니다.

또한 resume은 파이썬 함수의 멈춘 줄 바로 다음에서 계속되는 것이 아닙니다. **interrupt가 있던 Node를 처음부터 다시 실행**합니다.

아래 State의 `draft`, `decision`, `result`는 뒤 Node가 만드는 칸이므로 `NotRequired`로 표시합니다. 또한 router가 반환하는 `execute`·`reject`·`invalid`가 실제 Node 이름과 같아 Week 1의 path mapping을 생략한 축약형을 사용합니다. label과 Node 이름이 다르면 mapping 딕셔너리를 다시 전달해야 합니다.

In [ ]:
class ApprovalState(TypedDict):
    case_id: str
    request: str
    draft: NotRequired[str]
    decision: NotRequired[str]
    result: NotRequired[str]


ALLOWED_DECISIONS = ("승인", "거부")


def prepare_notice(state: ApprovalState):
    draft = f"[안내 초안] {state['request']} 담당 부서에서 확인 중입니다."
    return {"draft": draft}

In [ ]:
def review_notice(state: ApprovalState):
    # 이 출력은 최초 pause 때와 resume 때 각각 한 번 보입니다.
    print("review Node 시작:", state["case_id"])

    payload = {
        "question": "이 안내를 공식 발송 처리할까요?",
        "case_id": state["case_id"],
        "draft": state["draft"],
        "allowed_decisions": list(ALLOWED_DECISIONS),
    }
    decision = interrupt(payload)

    # payload의 allowed_decisions는 안내일 뿐이므로 코드로 검증합니다.
    validated = decision if decision in ALLOWED_DECISIONS else "입력오류"
    return {"decision": validated}

In [ ]:
def route_decision(
    state: ApprovalState,
) -> Literal["execute", "reject", "invalid"]:
    if state["decision"] == "승인":
        return "execute"
    if state["decision"] == "거부":
        return "reject"
    return "invalid"

In [ ]:
# 실제 DB 대신 처리된 operation ID를 메모리에 기록합니다.
processed_operation_ids: set[str] = set()


def execute_notice(state: ApprovalState):
    operation_id = f"{state['case_id']}:send-notice"
    if operation_id in processed_operation_ids:
        return {"result": "이미 처리된 작업이라 중복 실행하지 않았습니다."}

    processed_operation_ids.add(operation_id)
    return {"result": "✅ 승인되어 공식 안내 발송을 기록했습니다."}


def reject_notice(state: ApprovalState):
    return {"result": "⛔ 담당자가 거부하여 아무 작업도 하지 않았습니다."}


def invalid_notice(state: ApprovalState):
    return {"result": "⚠️ 허용되지 않은 결정이라 안전하게 중단했습니다."}

In [ ]:
approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("prepare", prepare_notice)
approval_builder.add_node("review", review_notice)
approval_builder.add_node("execute", execute_notice)
approval_builder.add_node("reject", reject_notice)
approval_builder.add_node("invalid", invalid_notice)

approval_builder.add_edge(START, "prepare")
approval_builder.add_edge("prepare", "review")
approval_builder.add_conditional_edges("review", route_decision)
approval_builder.add_edge("execute", END)
approval_builder.add_edge("reject", END)
approval_builder.add_edge("invalid", END)

approval_graph = approval_builder.compile(checkpointer=InMemorySaver())


def initial_case(case_id: str):
    return {
        "case_id": case_id,
        "request": "장학금 지급 예정일을 알려 주세요.",
    }

### 예측 1 · 거부 경로

담당자가 “거부”를 선택한다고 예상해 봅시다.

- 첫 invoke는 어느 Node에서 멈출까요?
- 멈춘 시점의 State에는 draft가 있을까요?
- <code>__interrupt__</code>는 저장 State에도 들어 있을까요?
- 거부 뒤 processed_operation_ids의 크기가 늘어날까요?

아래 두 셀은 **새 thread에서 pause한 뒤 같은 thread로 resume**합니다.

In [ ]:
reject_case_id = f"reject-{uuid.uuid4().hex[:8]}"
reject_config = {"configurable": {"thread_id": reject_case_id}}

reject_paused = approval_graph.invoke(
    initial_case(reject_case_id),
    reject_config,
)
reject_payload = reject_paused["__interrupt__"][0].value
reject_snapshot = approval_graph.get_state(reject_config)

print("질문:", reject_payload["question"])
print("초안:", reject_payload["draft"])
print("다음 Node:", reject_snapshot.next)

assert reject_snapshot.next == ("review",)
assert "__interrupt__" not in reject_snapshot.values

In [ ]:
side_effect_count_before = len(processed_operation_ids)
reject_result = approval_graph.invoke(
    Command(resume="거부"),
    reject_config,
)

print(reject_result["result"])

assert reject_result["decision"] == "거부"
assert len(processed_operation_ids) == side_effect_count_before
assert "아무 작업도 하지 않았습니다" in reject_result["result"]

### 거부 결과 설명

출력에서 “review Node 시작”이 pause 때 한 번, resume 때 한 번 나타납니다. 이것이 Node가 처음부터 재실행된다는 눈에 보이는 증거입니다. 이 print는 학습용 디버그 출력이라 위험하지 않지만, 같은 위치에 결제나 DB 추가가 있었다면 두 번 실행될 수 있습니다.

거부 경로에서는 execute Node로 가지 않았으므로 processed_operation_ids가 늘지 않았습니다. “거부도 명시적인 종료 경로를 가진다”는 점이 중요합니다. 승인만 설계하고 거부 경로를 누락하면 업무적으로 처리가 미완료되거나 그래프 실행 오류가 날 수 있습니다.

### 예측 2 · 승인 경로

이번에는 **앞의 거부 thread를 재사용하지 않고**, 새로운 case_id와 thread_id를 만듭니다. 이미 끝난 thread에 결정만 바꿔 넣는 것은 새 승인 실험이 아닙니다.

예상해 보세요.

1. 승인 뒤 실행되는 Node 이름은 무엇인가요?
2. processed_operation_ids에는 어떤 목적의 값이 추가될까요?
3. 같은 operation을 재시도하면 왜 중복 기록을 막을 수 있을까요?

In [ ]:
approve_case_id = f"approve-{uuid.uuid4().hex[:8]}"
approve_config = {"configurable": {"thread_id": approve_case_id}}

approve_paused = approval_graph.invoke(
    initial_case(approve_case_id),
    approve_config,
)
approve_payload = approve_paused["__interrupt__"][0].value
side_effect_count_before = len(processed_operation_ids)

approve_result = approval_graph.invoke(
    Command(resume="승인"),
    approve_config,
)

print("검토한 초안:", approve_payload["draft"])
print(approve_result["result"])

assert approve_result["decision"] == "승인"
assert len(processed_operation_ids) == side_effect_count_before + 1

### 승인 뒤에도 필요한 안전장치: idempotency

interrupt 뒤에 부수효과를 두는 것은 “승인받지 않은 실행”을 막습니다. 그러나 네트워크 재시도나 서버 장애 때문에 승인 뒤의 Node 자체가 다시 실행될 가능성까지 없애 주지는 않습니다.

**Idempotency(멱등성)**는 같은 operation을 여러 번 요청해도 외부 결과가 한 번 실행한 것과 같도록 만드는 성질입니다. 예제는 <code>case_id:send-notice</code>를 operation ID로 사용해 이미 처리한 작업인지 확인합니다. 실제 시스템에서는 메모리 set이 아니라 DB의 unique key나 외부 API의 idempotency key를 사용합니다.

따라서 안전한 설계는 두 층입니다.

1. interrupt로 **권한과 의사결정**을 확인한다.
2. operation ID로 **중복 실행**을 방지한다.

In [ ]:
# execute Node가 장애 뒤 재시도됐다고 가정한 직접 호출입니다.
count_before_retry = len(processed_operation_ids)
retry_result = execute_notice(approve_result)

print(retry_result["result"])

assert len(processed_operation_ids) == count_before_retry
assert "중복 실행하지 않았습니다" in retry_result["result"]

In [ ]:
# payload의 선택지 안내만 믿지 않고 잘못된 입력도 시험합니다.
invalid_case_id = f"invalid-{uuid.uuid4().hex[:8]}"
invalid_config = {"configurable": {"thread_id": invalid_case_id}}

approval_graph.invoke(initial_case(invalid_case_id), invalid_config)
count_before_invalid = len(processed_operation_ids)

invalid_result = approval_graph.invoke(
    Command(resume="보류"),
    invalid_config,
)

print("검증된 결정:", invalid_result["decision"])
print(invalid_result["result"])

assert invalid_result["decision"] == "입력오류"
assert len(processed_operation_ids) == count_before_invalid

### payload와 검증은 다르다

interrupt에 넣은 allowed_decisions는 화면이나 호출자에게 “승인/거부 중 고르세요”라고 알려 주는 데이터입니다. LangGraph가 그 목록을 보고 resume 값을 자동 검증하지는 않습니다. 실제 검증은 review_notice의 membership 조건이 수행합니다.

route_decision의 <code>Literal</code>도 개발 도구에 가능한 반환값을 알려 주는 타입 힌트이지, 실행 중의 보안 검사가 아닙니다. **힌트, UI 제한, 서버 검증을 서로 구분**해야 합니다.

#### 위험 코드 찾기

~~~python
def unsafe_review(state):
    student_system.append("발송 완료")  # 외부 상태 변경
    decision = interrupt("발송할까요?")
    return {"decision": decision}
~~~

<details>
<summary>무엇이 위험하고 어떻게 고칠까요?</summary>

resume할 때 unsafe_review가 처음부터 실행되므로 append가 반복될 수 있습니다. interrupt 앞에서는 초안 계산처럼 안전한 작업만 하고, 실제 append는 승인 분기 뒤의 별도 Node로 옮깁니다. 그 Node에도 operation ID를 사용한 중복 방지를 둡니다.

</details>

### 전이 과제 · 다른 학생지원 업무로 바꾸기

다음 중 하나를 골라 같은 구조로 설계하세요.

- 수강정원 초과 등록 요청
- 등록금 환불 요청
- 기숙사 입사 예외 요청

아래 빈칸을 채운 뒤 짝과 비교합니다.

| 항목 | 내 설계 |
|---|---|
| 업무 case를 구분할 thread_id | ________ |
| 최초 입력 State 칸 | ________ |
| 사람에게 보여 줄 interrupt payload | ________ |
| 허용할 결정과 검증 규칙 | ________ |
| 승인 뒤 부수효과 Node | ________ |
| 거부·잘못된 입력의 종료 경로 | ________ |
| 중복 방지 operation ID | ________ |

성공 기준은 코드를 길게 쓰는 것이 아니라, **승인 전 계산과 승인 후 부수효과의 경계가 분명한 것**입니다.

## 3-5. [보충] Streaming: 완료를 기다리지 않고 실행 과정을 관찰하기

HITL의 pause/resume을 이해한 뒤에 streaming을 봅니다. Streaming은 실행을 저장하는 기능도, 계산을 더 빠르게 만드는 기능도 아닙니다. **실행 중 만들어지는 중간 결과를 차례로 관찰하는 방법**입니다.

- updates: 각 Node가 이번 단계에서 새로 바꾼 값만 보여 줍니다.
- values: 입력 시점과 각 단계 뒤의 전체 State를 보여 줍니다.

Checkpointer와 streaming은 독립적입니다. 저장이 필요하면 Checkpointer를 붙이고, 관찰이 필요하면 stream을 사용합니다. 아래 결정적 그래프에는 Checkpointer와 LLM이 필요하지 않습니다.

In [ ]:
class TicketState(TypedDict):
    request: str
    status: NotRequired[str]
    owner: NotRequired[str]


def receive_ticket(state: TicketState):
    return {"status": "접수"}


def assign_ticket(state: TicketState):
    return {
        "status": "담당자 배정 완료",
        "owner": "학생지원팀",
    }

In [ ]:
ticket_builder = StateGraph(TicketState)
ticket_builder.add_node("receive", receive_ticket)
ticket_builder.add_node("assign", assign_ticket)
ticket_builder.add_edge(START, "receive")
ticket_builder.add_edge("receive", "assign")
ticket_builder.add_edge("assign", END)
ticket_graph = ticket_builder.compile()

ticket_input = {"request": "장학금 지급일 문의"}

### 실행 전 예측

그래프는 receive와 assign 두 Node를 실행합니다.

1. updates에서는 출력 묶음이 몇 개 나올까요?
2. receive의 updates에 request와 owner도 들어 있을까요?
3. values의 마지막 출력에는 어떤 세 칸이 들어 있을까요?
4. values에는 Node 실행 전의 최초 입력 State도 보일까요?

먼저 updates를 실행한 뒤 values와 비교하세요.

In [ ]:
update_parts = list(
    ticket_graph.stream(
        ticket_input,
        stream_mode="updates",
        version="v2",
    )
)

for part in update_parts:
    print(part["data"])

assert len(update_parts) == 2
assert update_parts[0]["data"] == {"receive": {"status": "접수"}}

In [ ]:
value_parts = list(
    ticket_graph.stream(
        ticket_input,
        stream_mode="values",
        version="v2",
    )
)

for part in value_parts:
    print(part["data"])

assert len(value_parts) == 3
assert value_parts[-1]["data"]["owner"] == "학생지원팀"
assert value_parts[-1]["data"]["status"] == "담당자 배정 완료"

### updates와 values 비교

| 모드 | 첫 출력 | Node 뒤 출력 | 적합한 용도 |
|---|---|---|---|
| updates | receive가 바꾼 status만 | assign이 바꾼 status와 owner만 | 어떤 Node가 무엇을 바꿨는지 디버깅 |
| values | 최초 입력 State 전체 | 매 단계 뒤 State 전체 | 화면에 현재 전체 상태를 표시 |

<code>version="v2"</code>의 각 stream part는 type, ns, data 같은 봉투 구조를 가집니다. 이번 실습에서는 실제 내용인 data만 읽습니다.

**주의:** values는 전체 State를 반복해서 보내므로 State가 크면 출력량이 커집니다. 필요한 변화만 관찰할 때는 updates가 더 간결합니다. 반대로 특정 시점의 완전한 화면이 필요하면 values가 편합니다.

### Streaming 빈칸과 예측 정답

1. Node별 ________ 만 보고 싶으면 updates를 사용합니다.
2. 각 단계의 ________ State를 보고 싶으면 values를 사용합니다.
3. Streaming은 실행 기록을 영구 ________ 하는 기능이 아닙니다.
4. Streaming을 사용한다고 전체 계산 시간이 자동으로 ________ 것은 아닙니다.

<details>
<summary>정답과 예측 결과 보기</summary>

빈칸: 1. 변경분, 2. 전체, 3. 저장, 4. 줄어드는

예측: updates는 두 Node에 대응해 두 묶음이 나옵니다. receive는 status만 바꾸므로 request와 owner는 그 updates에 없습니다. values는 최초 입력을 포함해 세 묶음이며, 마지막에는 request·status·owner가 모두 있습니다.

</details>

## 자주 생기는 오해 정리

| 오해 | 정확한 설명 |
|---|---|
| Reducer가 대화 기록을 디스크에 저장한다. | Reducer는 값 결합 규칙이고, 저장은 Checkpointer의 역할입니다. |
| DB Checkpointer이면 여러 thread가 자동으로 정보를 공유한다. | 지속성은 생기지만 State 범위는 여전히 thread별입니다. |
| InMemoryStore는 이름에 Store가 있으니 영구 저장된다. | cross-thread 공유는 가능하지만 RAM 기반이면 재시작 때 사라집니다. |
| interrupt의 allowed_decisions가 입력을 자동 검증한다. | payload는 안내 데이터이며 실제 조건문 검증이 필요합니다. |
| resume하면 interrupt 다음 줄에서 그대로 계속한다. | 해당 Node가 처음부터 재실행되고 interrupt가 resume 값을 반환합니다. |
| 승인 뒤 Node는 절대 두 번 실행되지 않는다. | 재시도 가능성을 고려해 idempotency가 필요합니다. |
| __interrupt__는 내가 만든 State 칸이다. | 대기 상태를 알리는 런타임의 특별 출력입니다. |
| Streaming을 켜면 작업 자체가 빨라진다. | 중간 결과를 빨리 볼 수 있을 뿐 총 계산량은 줄지 않을 수 있습니다. |

## 선택 부록 · 이번 주 필수 범위가 아닌 주제

다음 주제는 프로젝트에서 실제 필요가 생겼을 때 학습합니다. 이 노트북의 필수 퀴즈와 평가에는 포함하지 않습니다.

- **정적 breakpoint:** 특정 Node 앞이나 뒤에서 항상 멈추는 개발·디버깅 도구
- **update_state:** 저장된 State를 바탕으로 새 checkpoint를 만드는 편집 기능
- **Time travel:** 과거 checkpoint에서 replay하거나 새 분기를 만드는 기능
- **병렬 Node:** 같은 단계에서 여러 작업을 실행하고 Reducer로 결과를 합치는 설계
- **message token streaming:** LLM이 만드는 텍스트 조각을 UI에 즉시 표시하는 기능

특히 정적 breakpoint와 동적 interrupt를 같은 것으로 보지 마세요. 전자는 개발자가 정한 위치에서 디버깅하기 위한 멈춤이고, 후자는 실행 중 업무 조건에 따라 사람의 결정을 받기 위한 멈춤입니다.

## 용어 사전

| 용어 | 이 수업에서의 뜻 |
|---|---|
| State | 그래프가 현재 처리 중인 값들의 모음 |
| State update | Node가 반환한 변경분 |
| Node | State를 읽고 업데이트를 반환하는 한 업무 단계 |
| Edge | 다음 Node를 정하는 연결 |
| Reducer | 기존 값과 새 업데이트를 결합하는 규칙 |
| Checkpointer | 단계별 State 스냅샷을 저장하는 구성요소 |
| Checkpoint | 특정 시점의 저장된 State 스냅샷 |
| thread_id | 이어서 읽을 checkpoint 묶음을 구분하는 case ID |
| interrupt | 외부 입력을 기다리며 그래프를 일시정지하는 함수 |
| payload | interrupt가 호출자에게 보여 주는 질문과 문맥 데이터 |
| resume value | Command(resume=...)로 다시 전달하는 사람의 결정 |
| __interrupt__ | 그래프가 입력 대기 중임을 알리는 특별 출력 |
| side effect | 메일, 결제, DB 변경처럼 외부 세계를 바꾸는 행동 |
| idempotency | 같은 operation을 반복해도 결과가 중복되지 않는 성질 |
| updates stream | 각 Node의 변경분을 순서대로 관찰하는 모드 |
| values stream | 매 단계의 전체 State를 순서대로 관찰하는 모드 |

## 숙달 체크리스트

아래 항목을 코드 없이 설명하거나 직접 확인할 수 있으면 Week 3 핵심을 이수한 것입니다.

- [ ] Reducer와 Checkpointer의 차이를 학생지원 업무 비유로 설명한다.
- [ ] 같은 thread와 다른 thread의 결과를 예측한다.
- [ ] 메모리를 공유 범위와 지속성의 두 축으로 분류한다.
- [ ] pause 결과에서 payload를 읽고, __interrupt__가 State가 아님을 확인한다.
- [ ] 같은 thread_id와 Command(resume=...)로 중단된 실행을 재개한다.
- [ ] 승인, 거부, 잘못된 입력의 세 경로를 모두 안전하게 종료한다.
- [ ] interrupt 앞에 위험한 부수효과를 두지 않는다.
- [ ] 승인 뒤 작업에도 operation ID 기반 중복 방지가 필요한 이유를 말한다.

보충 Streaming까지 학습했다면 다음도 확인합니다.

- [ ] updates와 values 출력의 차이를 설명한다.

**Exit ticket:** “사람의 승인을 받는 것”과 “중복 실행을 막는 것”이 왜 서로 다른 안전 문제인지 세 문장으로 적으세요.

## [수업 후 과제] 3주 통합 팀 프로젝트 설계 틀

팀 프로젝트를 바로 코딩하기 전에 다음 한 장 설계서를 완성하세요.

### 1. 업무 문제

- 사용자와 담당자는 누구인가?
- 어떤 사건이 그래프를 시작하는가?
- 자동화의 성공 결과와 위험한 결과는 무엇인가?

### 2. State 표

| 칸 이름 | 타입 | 업무 의미 | 업데이트 방식 | 민감정보 여부 |
|---|---|---|---|---|
| ________ | ________ | ________ | 덮어쓰기 / Reducer | 예 / 아니오 |

### 3. Node·Edge 설계

| Node | 읽는 State | 반환하는 업데이트 | 부수효과 | 실패 시 경로 |
|---|---|---|---|---|
| ________ | ________ | ________ | 없음 / 있음 | ________ |

Edge 그림에는 START, 정상 종료, 거부 종료, 오류 종료, 반복 시 종료 조건을 모두 표시합니다.

### 4. Memory와 HITL

- thread_id는 무엇을 case로 구분하는가?
- cross-thread로 공유할 정보가 정말 필요한가?
- RAM과 DB 중 어떤 지속성이 필요한가?
- interrupt payload에 어떤 업무 문맥을 보여 줄 것인가?
- resume 값을 어디서 검증할 것인가?
- 부수효과 operation ID는 어떻게 만들 것인가?

### 5. 최소 테스트

정상 승인, 거부, 잘못된 결정, 다른 thread 격리, 같은 operation 재시도를 각각 한 번 검사합니다.

## [수업 후 과제] 팀 프로젝트 루브릭 · 20점

| 영역 | 4점 기준 |
|---|---|
| 업무 문제와 범위 | 사용자, 시작 사건, 성공 결과, 위험 행동이 구체적이다. |
| State와 Memory | State 칸·타입·Reducer가 타당하고, thread/cross-thread 및 RAM/DB 선택 이유가 있다. |
| Node와 Edge | Node 책임이 작고 명확하며 모든 분기에 종료 또는 다음 단계가 있다. |
| HITL와 안전 | 승인 지점, 입력 검증, 부수효과 위치, idempotency 전략이 명시되어 있다. |
| 검증과 설명 | 정상·거부·오류·thread 격리·중복 재시도 결과를 구조적 출력과 assert로 확인한다. |

각 영역은 4점 “명확하고 검증됨”, 3점 “대체로 맞지만 일부 근거 부족”, 2점 “구성은 있으나 혼동 존재”, 1점 “이름만 제시”, 0점 “누락”으로 평가합니다.

긴 코드나 화려한 LLM 답변은 점수 기준이 아닙니다. **업무 흐름, State 변화, 안전 경계, 검증 근거를 다른 팀이 읽고 재현할 수 있는가**가 기준입니다.

## Week 3 최종 정리

~~~text
Node 업데이트
   ↓ Reducer가 현재 State를 만듦
   ↓ Checkpointer가 thread별 checkpoint를 저장
   ↓ interrupt가 사람의 결정을 기다림
   ↓ 검증된 resume 값으로 안전한 Edge를 선택
   ↓ 승인 뒤 idempotent한 부수효과 실행
   ↓ (보충) updates 또는 values로 과정을 관찰
~~~

### 최종 이해 확인

1. Reducer와 Checkpointer는 각각 어떤 질문에 답하나요?
2. DB에 저장되는 Checkpointer이면 자동으로 cross-thread memory가 되나요?
3. __interrupt__와 저장 State는 어떻게 구분하나요?
4. resume 때 interrupt Node의 코드는 어디서부터 실행되나요?
5. allowed_decisions payload만으로 입력 검증이 끝나나요?
6. 위험한 DB 추가 Node는 interrupt 전과 후 중 어디에 둬야 하나요?
7. (보충) 변경분만 보고 싶을 때 어떤 stream mode를 쓰나요?

<details>
<summary>정답 보기</summary>

1. Reducer는 값을 어떻게 합칠지, Checkpointer는 단계별 State를 어떻게 저장할지 답합니다.
2. 아닙니다. 지속성과 공유 범위는 별개의 축입니다.
3. __interrupt__는 호출 결과의 특별 출력이고, 저장 State는 get_state(config).values에서 봅니다.
4. interrupt가 있던 Node의 처음부터 다시 실행됩니다.
5. 아닙니다. resume 값을 서버 코드에서 검증해야 합니다.
6. 승인 뒤의 별도 Node에 두고 idempotency도 적용합니다.
7. updates를 사용합니다.

</details>